# EDA · physical_vendas_caixa 

| Seção | Pergunta |
|---|---|
| 1 | A tabela está íntegra? (nulos, duplicados, valores e datas inválidas, dias sem venda) |
| 2 | Como se distribui o valor da venda e onde estão os outliers? |
| 3 | Como o cliente paga e quanto ele se identifica (CPF)? |
| 4 | Como as lojas, caixas e operadores se comparam? |
| 5 | Quais padrões de tempo existem? (dia, mês, sazonalidade, semana, hora, decomposição) |
| 6 | Como se comporta a série mensal de cada loja? (base do P3) |
| 7 | Resumo dos achados |
| 8 | Conclusões e decisões para a engenharia de features |

## 0. Configuração

In [0]:
%pip install python-dotenv statsmodels --quiet

In [0]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from pyspark.sql import DataFrame, functions as F, types as T
from statsmodels.tsa.seasonal import seasonal_decompose

load_dotenv("/Workspace/Users/eng.stephanni@gmail.com/.env", override=True)

CONTA = os.environ["STORAGE_ACCOUNT_NAME"]
HOST = f"{CONTA}.dfs.core.windows.net"
AUTH = {
    f"fs.azure.account.auth.type.{HOST}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{HOST}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{HOST}": os.environ["ADLS_CLIENT_ID"],
    f"fs.azure.account.oauth2.client.secret.{HOST}": os.environ["ADLS_CLIENT_SECRET"],
    f"fs.azure.account.oauth2.client.endpoint.{HOST}": f"https://login.microsoftonline.com/{os.environ['ADLS_TENANT_ID']}/oauth2/token",
}

# Onde está a tabela da Silver (container, caminho) — Delta. Lojas só para trazer o nome da loja.
SILVER = {
    "physical_vendas_caixa": ("raw", "silver/physical_vendas_caixa"),
    "physical_lojas": ("squad3", "silver/physical_lojas"),
}
DESTINO = ("squad3", "ml")   # saída das próximas etapas (nunca a Silver)


def url(container: str, caminho: str) -> str:
    return f"abfss://{container}@{HOST}/{caminho}"


def ler(tabela: str) -> DataFrame:
    """Lê uma tabela da Silver."""
    return spark.read.format("delta").options(**AUTH).load(url(*SILVER[tabela]))


def salvar(df: DataFrame, nome: str, particoes: list = None) -> None:
    """Grava em squad3/ml/<nome> (Delta). Bloqueia qualquer gravação na Silver."""
    if "silver" in nome.lower():
        raise ValueError("A Silver é somente leitura.")
    w = df.write.format("delta").options(**AUTH).mode("overwrite").option("overwriteSchema", "true")
    (w.partitionBy(*particoes) if particoes else w).save(url(DESTINO[0], f"{DESTINO[1]}/{nome}"))


def perfil(df: DataFrame) -> pd.DataFrame:
    """Tipo, nulos e distintos de cada coluna."""
    n = df.count()
    agg = df.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(f"n_{c}") for c in df.columns],
                 *[F.approx_count_distinct(c).alias(f"d_{c}") for c in df.columns]).first()
    return pd.DataFrame({
        "coluna": df.columns,
        "tipo": [f.dataType.simpleString() for f in df.schema],
        "nulos": [agg[f"n_{c}"] for c in df.columns],
        "pct_nulos": [round(100 * agg[f"n_{c}"] / n, 2) for c in df.columns],
        "distintos": [agg[f"d_{c}"] for c in df.columns],
    })


def outliers_iqr(df: DataFrame, coluna: str) -> dict:
    """Limites de Tukey (1,5 × IQR) e quantidade de valores fora deles."""
    q1, q3 = df.approxQuantile(coluna, [0.25, 0.75], 0.001)
    lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    return {"coluna": coluna, "q1": q1, "q3": q3, "limite_inf": lo, "limite_sup": hi,
            "fora": df.filter((F.col(coluna) < lo) | (F.col(coluna) > hi)).count()}


def grafico_distribuicao(df: DataFrame, coluna: str) -> None:
    """Histograma + boxplot de uma coluna numérica."""
    s = df.select(F.col(coluna).cast("double")).dropna().toPandas()[coluna]
    fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3))
    a.hist(s, bins=50, color=AZUL); a.set_title(f"Distribuição · {coluna}")
    b.boxplot(s, vert=False, flierprops={"markersize": 2}); b.set_title(f"Boxplot · {coluna}")
    plt.tight_layout(); plt.show()


def mapa_calor(tabela: pd.DataFrame, titulo: str, cmap: str = "Blues", **kw) -> None:
    """Mapa de calor de uma tabela pandas (linhas × colunas)."""
    plt.figure(figsize=(12, max(3, 0.28 * len(tabela))))
    plt.imshow(tabela.astype(float), aspect="auto", cmap=cmap, **kw); plt.colorbar(); plt.grid(False)
    plt.yticks(range(len(tabela)), tabela.index)
    plt.xticks(range(tabela.shape[1]), tabela.columns, rotation=90 if tabela.shape[1] > 24 else 0)
    plt.title(titulo); plt.tight_layout(); plt.show()


plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#E6EAEE"})
AZUL, OURO = "#1F4E8C", "#D99A00"
DIAS = ["dom", "seg", "ter", "qua", "qui", "sex", "sáb"]   # F.dayofweek: 1 = domingo
V = "valor_total_venda"

## 1. Estrutura e qualidade

In [0]:
bruto = ler("physical_vendas_caixa")
n = bruto.count()
print(f"{n:,} linhas · {len(bruto.columns)} colunas")
display(perfil(bruto))

# Cópia em memória para a análise (a Silver não é alterada)
cpf_ok = F.length(F.regexp_replace(F.coalesce(F.col("cpf_cliente").cast("string"), F.lit("")), "[^0-9]", "")) == 11
df = (bruto.withColumn("ts", F.to_timestamp("dt_venda"))
           .withColumn(V, F.col(V).cast("double"))
           .withColumn("cpf_valido", cpf_ok)
           .drop("cpf_cliente"))                      # LGPD: o CPF não aparece nas saídas

# Nome da loja (sem o prefixo "Loja ") para os gráficos; id sem cadastro aparece como "id N (sem cadastro)"
nomes = (ler("physical_lojas").select("id_loja", F.regexp_replace("nome_loja", r"^Loja\s+", "").alias("nome_loja"))
                              .dropDuplicates(["id_loja"]))
df = (df.join(F.broadcast(nomes), "id_loja", "left")
        .withColumn("loja", F.coalesce("nome_loja", F.concat(F.lit("id "), F.col("id_loja").cast("string"), F.lit(" (sem cadastro)"))))
        .drop("nome_loja"))

checagens = {
    "id_transacao duplicado": n - df.select("id_transacao").distinct().count(),
    "linhas 100% duplicadas": n - bruto.distinct().count(),
    "valor nulo ou <= 0": df.filter(F.col(V).isNull() | (F.col(V) <= 0)).count(),
    "dt_venda nula ou inválida": df.filter(F.col("ts").isNull()).count(),
    "dt_venda no futuro": df.filter(F.col("ts") > F.current_timestamp()).count(),
    "id_loja / id_caixa / id_operador nulo": df.filter(F.col("id_loja").isNull() | F.col("id_caixa").isNull()
                                                       | F.col("id_operador").isNull()).count(),
}
display(pd.DataFrame(checagens.items(), columns=["checagem", "linhas"]))

ini, fim = df.agg(F.min("ts"), F.max("ts")).first()
com_venda = {r.d for r in df.select(F.to_date("ts").alias("d")).distinct().collect()}
sem_venda = sorted(set(pd.date_range(ini.date(), fim.date()).date) - com_venda)
print(f"Período: {ini:%d/%m/%Y} a {fim:%d/%m/%Y} · dias com venda: {len(com_venda):,} · dias sem venda: {len(sem_venda)}")
print("Dias sem venda:", [d.strftime("%d/%m/%Y") for d in sem_venda])

## 2. Valor da venda: distribuição e outliers

In [0]:
display(df.select(V).summary("count", "min", "1%", "5%", "25%", "50%", "75%", "95%", "99%", "max", "mean", "stddev"))

o = outliers_iqr(df, V)
display(pd.DataFrame([o]))
print(f"{o['fora'] / n:.2%} das vendas estão fora dos limites de Tukey")
grafico_distribuicao(df, V)

# Alguma loja concentra valores extremos?
fora = ((F.col(V) < o["limite_inf"]) | (F.col(V) > o["limite_sup"])).cast("int")
display(df.groupBy("loja").agg(F.count("*").alias("vendas"), F.round(100 * F.avg(fora), 2).alias("pct_outliers"))
          .orderBy(F.desc("pct_outliers")))

print("10 maiores vendas:")
display(df.orderBy(F.desc(V)).limit(10))

## 3. Pagamento e identificação do cliente

In [0]:
def resumo(chave: str) -> pd.DataFrame:
    """Transações, receita, ticket médio e participação (%) por categoria."""
    p = df.groupBy(chave).agg(F.count("*").alias("transacoes"), F.sum(V).alias("receita"),
                              F.avg(V).alias("ticket_medio")).toPandas()
    p["pct_transacoes"] = 100 * p.transacoes / p.transacoes.sum()
    p["pct_receita"] = 100 * p.receita / p.receita.sum()
    return p.sort_values("receita").round(2)


pag = resumo("tipo_pagamento")
display(pag)
fig, ax = plt.subplots(1, 2, figsize=(13, 3))
pag.plot.barh(x="tipo_pagamento", y=["pct_transacoes", "pct_receita"], ax=ax[0], color=[AZUL, OURO], title="Participação (%)")
pag.plot.barh(x="tipo_pagamento", y="ticket_medio", ax=ax[1], color=AZUL, legend=False, title="Ticket médio (R$)")
plt.tight_layout(); plt.show()

print("Cliente identificado (CPF válido) compra mais?")
display(resumo("cpf_valido"))

cpf_loja = (df.groupBy("loja").agg(F.round(100 * F.avg(F.col("cpf_valido").cast("int")), 1).alias("pct_cpf"))
              .orderBy("pct_cpf").toPandas())
cpf_loja.plot.barh(x="loja", y="pct_cpf", color=AZUL, legend=False, figsize=(9, 8), title="% de vendas com CPF por loja")
plt.show()

## 4. Lojas, caixas e operadores

In [0]:
lojas = (df.groupBy("id_loja", "loja").agg(F.count("*").alias("transacoes"), F.sum(V).alias("receita"),
                                   F.avg(V).alias("ticket_medio"), F.countDistinct("id_caixa").alias("caixas"),
                                   F.countDistinct("id_operador").alias("operadores"),
                                   F.countDistinct(F.to_date("ts")).alias("dias_com_venda"))
           .toPandas().sort_values("receita"))
lojas["pct_receita"] = 100 * lojas.receita / lojas.receita.sum()
lojas["transacoes_por_caixa"] = lojas.transacoes / lojas.caixas
display(lojas.round(2))

fig, ax = plt.subplots(1, 3, figsize=(17, 8), sharey=True)
for eixo, (col, tit, cor) in zip(ax, [("receita", "Receita total (R$)", AZUL), ("ticket_medio", "Ticket médio (R$)", OURO),
                                      ("transacoes_por_caixa", "Transações por caixa", AZUL)]):
    eixo.barh(lojas.loja, lojas[col], color=cor); eixo.set_title(tit)
plt.tight_layout(); plt.show()

print("Spearman transações × ticket médio (por loja):", round(lojas.transacoes.corr(lojas.ticket_medio, method="spearman"), 2))

op = df.groupBy("id_operador").agg(F.countDistinct("id_loja").alias("lojas"), F.count("*").alias("transacoes"))
print("Operadores que aparecem em mais de uma loja:", op.filter("lojas > 1").count(), "de", op.count())

print("Transações por caixa em um dia:")
caixa_dia = df.groupBy("id_loja", "id_caixa", F.to_date("ts").alias("dia")).count()
display(caixa_dia.select("count").summary("min", "25%", "50%", "75%", "95%", "max", "mean"))

## 5. Padrões no tempo

In [0]:
t = (df.withColumn("dia", F.to_date("ts")).withColumn("mes", F.trunc("ts", "month"))
       .withColumn("dia_semana", F.dayofweek("ts")).withColumn("hora", F.hour("ts")))

# 5.1 Receita diária com média móvel de 7 dias
diaria = t.groupBy("dia").agg(F.sum(V).alias("receita"), F.count("*").alias("transacoes")).toPandas()
diaria = diaria.set_index(pd.to_datetime(diaria.dia)).drop(columns="dia").sort_index()
plt.figure(figsize=(12, 3.5))
plt.plot(diaria.index, diaria.receita / 1e3, color=AZUL, alpha=0.35, lw=0.8, label="diária")
plt.plot(diaria.index, diaria.receita.rolling(7).mean() / 1e3, color=OURO, lw=1.8, label="média móvel 7 dias")
plt.title("Receita diária da rede (R$ mil)"); plt.legend(); plt.show()
print("10 dias de maior receita:"); display(diaria.nlargest(10, "receita").round(2).reset_index())

In [0]:
# 5.2 Receita, transações e ticket por mês (variação mês a mês e ano a ano)
mensal = t.groupBy("mes").agg(F.sum(V).alias("receita"), F.count("*").alias("transacoes"), F.avg(V).alias("ticket"),
                              F.countDistinct("dia").alias("dias_com_venda")).toPandas()
mensal = mensal.set_index(pd.to_datetime(mensal.mes)).drop(columns="mes").sort_index()
mensal["var_mom_pct"] = 100 * mensal.receita.pct_change()
mensal["var_yoy_pct"] = 100 * mensal.receita.pct_change(12)
display(mensal.round(2).reset_index())

incompletos = mensal[mensal.dias_com_venda < mensal.index.days_in_month]
print("Meses com dias sem venda (podem estar incompletos e distorcer o modelo):", list(incompletos.index.strftime("%Y-%m")))

fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
for eixo, (col, tit) in zip(ax, [("receita", "Receita (R$)"), ("transacoes", "Transações"), ("ticket", "Ticket médio (R$)")]):
    eixo.plot(mensal.index, mensal[col], color=AZUL, marker="o", ms=3); eixo.set_title(f"{tit} por mês")
    eixo.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

In [0]:
# 5.3 Sazonalidade: mesmo mês em anos diferentes, índice sazonal e autocorrelação
por_ano = mensal.receita.groupby([mensal.index.month, mensal.index.year]).sum().unstack()
(por_ano / 1e6).plot(marker="o", figsize=(10, 3.5), title="Receita por mês do ano, uma linha por ano (R$ mi)")
plt.xticks(range(1, 13)); plt.show()

indice = mensal.receita.groupby(mensal.index.month).mean() / mensal.receita.mean()
acf = pd.Series({lag: mensal.receita.autocorr(lag) for lag in range(1, 13)})
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
indice.plot.bar(ax=ax[0], color=AZUL, title="Índice sazonal (média do mês ÷ média geral)"); ax[0].axhline(1, color="k", lw=0.6)
acf.plot.bar(ax=ax[1], color=OURO, title="Autocorrelação da receita mensal (lag em meses)"); ax[1].axhline(0, color="k", lw=0.6)
plt.tight_layout(); plt.show()

In [0]:
# 5.4 Semana e hora (médias por dia, para não favorecer dias que aparecem mais vezes)
sem = (t.groupBy("dia", "dia_semana").agg(F.sum(V).alias("receita"), F.count("*").alias("transacoes"))
        .groupBy("dia_semana").agg(F.avg("receita").alias("receita_media_dia"), F.avg("transacoes").alias("transacoes_media_dia"))
        .orderBy("dia_semana").toPandas())
sem["dia_semana"] = [DIAS[d - 1] for d in sem.dia_semana]
hora = t.groupBy("hora").agg(F.count("*").alias("transacoes"), F.avg(V).alias("ticket")).orderBy("hora").toPandas()
display(sem.round(2)); display(hora.round(2))

fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
ax[0].bar(sem.dia_semana, sem.receita_media_dia / 1e3, color=AZUL); ax[0].set_title("Receita média por dia da semana (R$ mil)")
ax[1].bar(hora.hora, hora.transacoes, color=AZUL); ax[1].set_title("Transações (barras) e ticket médio (linha) por hora")
tk = ax[1].twinx(); tk.plot(hora.hora, hora.ticket, color=OURO, marker="o", ms=3); tk.grid(False)
plt.tight_layout(); plt.show()

calor = t.groupBy("dia_semana", "hora").count().toPandas().pivot(index="dia_semana", columns="hora", values="count").fillna(0)
calor.index = [DIAS[d - 1] for d in calor.index]
mapa_calor(calor, "Transações: dia da semana × hora")

pag_hora = t.groupBy("hora").pivot("tipo_pagamento").count().orderBy("hora").toPandas().set_index("hora").fillna(0)
(100 * pag_hora.div(pag_hora.sum(axis=1), axis=0)).plot.area(figsize=(10, 3.5), title="Forma de pagamento por hora (% das transações)")
plt.show()

### 5.5 Decomposição da série da rede (tendência + sazonalidade + resíduo)

- **Força da tendência / da sazonalidade** (0 a 1): perto de 1 = componente forte e claro; perto de 0 = quase só ruído.
- **Aditivo × multiplicativo**: o que deixa o menor resíduo relativo descreve melhor a série. Multiplicativo = a amplitude sazonal cresce junto com o nível da receita.
  Isso define o tipo de sazonalidade do Holt-Winters no P3.

In [0]:
def forca(componente: pd.Series, resid: pd.Series) -> float:
    """Força do componente (Hyndman): 1 − var(resíduo) / var(componente + resíduo)."""
    return max(0.0, 1 - resid.var() / (componente + resid).var())


def forca_sazonal(s: pd.Series) -> float:
    """Força da sazonalidade de uma série mensal (precisa de pelo menos 24 meses)."""
    s = s.dropna()
    if len(s) < 24:
        return np.nan
    d = seasonal_decompose(np.log(s.asfreq("MS")), period=12)
    return forca(d.seasonal, d.resid)


rede = mensal.receita.asfreq("MS")
dec = {m: seasonal_decompose(rede, model=m, period=12) for m in ("additive", "multiplicative")}
log_dec = seasonal_decompose(np.log(rede), period=12)          # multiplicativo = aditivo no log

tend = dec["additive"].trend.dropna()
decomp = {
    "força da tendência (0–1)": round(forca(log_dec.trend, log_dec.resid), 2),
    "força da sazonalidade (0–1)": round(forca(log_dec.seasonal, log_dec.resid), 2),
    "variação da tendência no período": f"{100 * (tend.iloc[-1] / tend.iloc[0] - 1):+.1f}%",
    "resíduo relativo · aditivo": f"{100 * dec['additive'].resid.std() / rede.mean():.2f}%",
    "resíduo relativo · multiplicativo": f"{100 * (dec['multiplicative'].resid - 1).std():.2f}%",
}
modelo_sugerido = "multiplicativo" if (dec["multiplicative"].resid - 1).std() < dec["additive"].resid.std() / rede.mean() else "aditivo"
decomp["sazonalidade sugerida para o Holt-Winters"] = modelo_sugerido
display(pd.DataFrame(decomp.items(), columns=["medida", "valor"]).astype(str))

fig = dec["multiplicative" if modelo_sugerido == "multiplicativo" else "additive"].plot()
fig.set_size_inches(12, 8); fig.suptitle(f"Decomposição {modelo_sugerido} da receita mensal da rede", y=1.01)
plt.tight_layout(); plt.show()

## 6. Série mensal por loja (base do P3)

- `cv_pct`: volatilidade da loja (desvio ÷ média). Quanto maior, mais difícil de prever.
- `corr_lag12`: correlação com o mesmo mês do ano anterior. Alta = sazonalidade forte, o que favorece o baseline sazonal e o Holt-Winters.
- `forca_sazonal`: força da sazonalidade da loja na decomposição (0 a 1).
- `quartil`: Q1 são as lojas pequenas, candidatas a serem agrupadas no modelo.

In [0]:
loja_mes = (t.groupBy("loja", "mes").agg(F.sum(V).alias("receita")).toPandas()
              .pivot(index="mes", columns="loja", values="receita").sort_index())
loja_mes.index = pd.to_datetime(loja_mes.index)
print(f"{loja_mes.shape[1]} lojas · {loja_mes.shape[0]} meses · loja-mês sem venda: {int(loja_mes.isna().sum().sum())}")

serie = pd.DataFrame({
    "receita_total": loja_mes.sum(),
    "media_mensal": loja_mes.mean(),
    "cv_pct": 100 * loja_mes.std() / loja_mes.mean(),
    "meses": loja_mes.notna().sum(),
    "corr_lag1": loja_mes.apply(lambda s: s.autocorr(1)),
    "corr_lag12": loja_mes.apply(lambda s: s.autocorr(12)),
    "forca_sazonal": loja_mes.apply(forca_sazonal),
})
serie["pct_receita"] = 100 * serie.receita_total / serie.receita_total.sum()
serie["quartil"] = pd.qcut(serie.receita_total, 4, labels=["Q1 (menores)", "Q2", "Q3", "Q4 (maiores)"])
display(serie.sort_values("receita_total").round(3).reset_index())
display(serie.groupby("quartil", observed=True).agg(lojas=("receita_total", "size"), pct_receita=("pct_receita", "sum"),
                                                    cv_medio=("cv_pct", "mean"), corr_lag12_media=("corr_lag12", "mean")).round(2).reset_index())
print("Lojas pequenas (Q1):", sorted(serie[serie.quartil == "Q1 (menores)"].index.tolist()))

fig, ax = plt.subplots(1, 2, figsize=(18, 6))
(loja_mes / 1e3).plot(ax=ax[0], legend=False, alpha=0.5, title="Receita mensal por loja (R$ mil)")
ax[1].scatter(serie.media_mensal / 1e3, serie.cv_pct, color=AZUL)
for loja, r in serie.iterrows():
    ax[1].annotate(loja, (r.media_mensal / 1e3, r.cv_pct), fontsize=7)
ax[1].set(title="Tamanho × volatilidade", xlabel="receita média mensal (R$ mil)", ylabel="CV (%)")
plt.tight_layout(); plt.show()

idx = (loja_mes / loja_mes.mean()).T
idx.columns = idx.columns.strftime("%Y-%m")
mapa_calor(idx, "Receita do mês ÷ média da loja (o padrão sazonal se repete entre as lojas?)", cmap="RdBu_r", vmin=0.6, vmax=1.4)

## 7. Resumo dos achados

In [0]:
achados = {
    "linhas": f"{n:,}",
    "período": f"{ini:%d/%m/%Y} a {fim:%d/%m/%Y}",
    "dias sem venda": len(sem_venda),
    "problemas de qualidade (soma das checagens)": sum(checagens.values()),
    "vendas fora dos limites de Tukey": f"{o['fora'] / n:.2%}",
    "vendas com CPF": f"{df.filter('cpf_valido').count() / n:.2%}",
    "lojas / meses": f"{loja_mes.shape[1]} / {loja_mes.shape[0]}",
    "meses possivelmente incompletos": ", ".join(incompletos.index.strftime("%Y-%m")) or "nenhum",
    "autocorrelação lag 12 da rede": round(acf[12], 2),
    "força da tendência / sazonalidade da rede": f"{decomp['força da tendência (0–1)']} / {decomp['força da sazonalidade (0–1)']}",
    "sazonalidade sugerida para o Holt-Winters": modelo_sugerido,
    "receita das lojas Q1": f"{serie[serie.quartil == 'Q1 (menores)'].pct_receita.sum():.1f}% do total",
}
display(pd.DataFrame(achados.items(), columns=["achado", "valor"]).astype(str))


## 8. Conclusões e decisões para a engenharia de features

**1. Qualidade dos dados** (seção 1)
- Problemas encontrados: nenhum. As 375.000 vendas passaram em todas as checagens: sem id_transacao duplicado, sem linhas repetidas, sem valor nulo ou ≤ 0, sem data inválida ou no futuro e sem ids de loja, caixa ou operador nulos. (O perfil mostra 369.956 ids distintos e 30 lojas porque usa contagem aproximada; a checagem exata confirma 375.000 vendas únicas em 29 lojas.)
- CPF: 55,5% das vendas não têm CPF (nulo), o que é esperado: o cliente informa o CPF se quiser.
- Dias sem venda: nenhum na rede (974 dias, de 01/01/2024 a 31/08/2026). Quatro lojas pequenas têm 1 ou 2 dias sem venda (Taparuba e Cunhambebe 972 dias, Diadema e Pontal 973), sem impacto na série mensal: nenhuma loja tem mês sem venda.
- Tratamento proposto: nenhum. A série de 29 lojas × 32 meses entra inteira no modelo.

**2. Valor da venda** (seção 2)
- Mediana de R$ 254,94 e média de R$ 365,91; 99% das vendas ficam abaixo de R$ 1.664 e a maior é R$ 4.378,14.
- 5,28% das vendas (19.813) passam do limite de Tukey (R$ 1.032,23), todas pelo lado de cima: é a cauda natural de compras grandes, e não erro. → **manter**.
- Os outliers estão espalhados por igual entre as lojas (de 4,9% a 5,9%), então nenhuma loja concentra valores extremos.

**3. Pagamento e cliente** (seção 3)
- Cartão domina: 60,1% da receita e das transações; Dinheiro e Pix ficam com ~20% cada. O ticket é praticamente igual nas três formas (R$ 365 a R$ 366).
- 44,5% das vendas têm CPF. O cliente identificado não compra mais: ticket de R$ 366,19 com CPF contra R$ 365,68 sem.

**4. Lojas e operação** (seção 4)
- Maiores lojas em receita: Duque de Caxias – Jardim Primavera (R$ 8,48 mi, 6,2% da rede), Belford Roxo – Nova Aurora (R$ 8,07 mi) e Itapecerica (R$ 8,05 mi). Menores: Diadema – Canhema (R$ 2,32 mi), Taparuba (R$ 2,34 mi) e Cunhambebe – Nova Angra (R$ 2,34 mi).
- O ticket médio é quase igual em todas as lojas (R$ 359 a R$ 372): a diferença de receita vem do número de transações (6 mil a 23 mil por loja). Spearman transações × ticket: −0,47.
- Todas as lojas têm 5 caixas. Cada caixa faz em média 3 vendas por dia (máximo de 20), sem sinal de sobrecarga na operação.
- Operadores: os ids vão de 1 a 20 e se repetem em todas as lojas, ou seja, o `id_operador` é numerado por loja, não é único na rede. Para identificar um operador é preciso usar o par (`id_loja`, `id_operador`).

**5. Série da rede** (seções 5.1 a 5.5)
- Meses incompletos: nenhum → a série de 32 meses entra inteira no treino.
- Tendência: praticamente estável (força 0,22; −0,7% no período). A variação ano contra ano fica entre −5% e +3% em todos os meses.
- Força da sazonalidade: 1,00, e a autocorrelação com o mesmo mês do ano anterior também é 1,00: o padrão anual se repete quase igual todo ano.
- Pico: dezembro (~R$ 6,9 mi, 61% acima da média) e novembro (27% acima); os 10 dias de maior receita são todos de dezembro. Baixa: fevereiro (~R$ 2,9 mi, 32% abaixo da média), janeiro e julho.
- Semana: domingo tem a maior receita média por dia (R$ 162,9 mil) e sábado a menor (R$ 122,5 mil).
- Hora: as lojas vendem 24 horas. O movimento é baixo na madrugada (0h–4h), tem um pico no almoço (12h–13h) e o maior pico à noite (18h–22h, com máximo às 21h). O ticket não muda ao longo do dia.

**6. Séries por loja** (seção 6)
- Todas as 29 lojas têm os 32 meses completos.
- Lojas pequenas (Q1) que serão agrupadas: Diadema – Canhema, Taparuba, Cunhambebe – Nova Angra, Pontal – Conjunto José Pedro Carolo, Monte Azul Paulista – Jardim Amazonas, Bom Sucesso (id 11), Divino de São Lourenço e Itarana (id 15). Representam 15,2% da receita e são exatamente as lojas com `peso_vendas` 3 e 4.
- Volatilidade: o CV vai de 18,8% a 26,4%; a mais volátil é Taparuba (26,4%), seguida de Bom Sucesso e Divino de São Lourenço (24,6%), todas do Q1, o que reforça o agrupamento.
- Sazonalidade por loja: forte em todas (`forca_sazonal` de 0,88 a 1,00; `corr_lag12` de 0,77 a 0,96) e um pouco maior nas lojas grandes (média 0,93 no Q4 contra 0,85 no Q1). As mais fracas: Monte Azul Paulista (0,77 / 0,88), Porto das Caixas (0,77) e Pontal (0,82).
- Lojas sem cadastro em physical_lojas: 7 (Caçapava), 11 (Bom Sucesso), 15 (Itarana) e 29 (Campos Elyseos), que